[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 中等：批归一化（BatchNorm）

实现**批归一化（Batch Normalization）**，同时支持**训练**和**推理**模式。

训练时使用**当前批次的统计量**，并更新运行统计量：

$$
\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta
$$

其中，$\mu_B$ 和 $\sigma_B^2$ 是沿**批次维度**（dim=0）计算的均值和方差。

推理时使用传入的**运行均值与方差**，替代当前批次的统计量。

### 函数或类接口
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — 对批次中所有样本的每个特征进行归一化
    # running_mean、running_var：训练时原地更新，推理时直接使用
```

### 约束条件
- **禁止**使用 `F.batch_norm`、`nn.BatchNorm1d` 等内置实现。
- 沿 `dim=0` 计算批次均值和方差，方差计算设置 `unbiased=False`。
- 按 PyTorch 的方式更新运行统计量：`running = (1 - momentum) * running + momentum * batch_stat`。
- 当 `training=False` 时，使用 `running_mean` / `running_var` 进行推理。
- 必须支持对 `x`、`gamma` 和 `beta` 自动求导；运行统计量应视为缓冲区，不应作为需要梯度的参数。

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ 在此编写你的实现

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    pass  # 在此填写实现

In [ ]:
# 🧪 调试
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# 运行统计量通常与特征位于同一设备，并匹配特征维度
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# 训练模式：使用批次统计量，并更新 running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # 应为 ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # 应为 ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# 推理模式：仅使用 running_mean / running_var
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

In [ ]:
# ✅ 提交
from torch_judge import check
check("batchnorm")